# 02.1 — RAG Baseline Evaluation (Groq API — Llama 4 Scout)

Versi notebook 02 yang menggunakan **Groq API** dengan model **Llama 4 Scout** (17B MoE, 109B total).

**Pipeline:**
```
Query → BM25 Retrieval (top-5) → LLM Generate → Extract Label
```

**Model:** `meta-llama/llama-4-scout-17b-16e-instruct` via Groq API
**Free tier:** 500K token/hari, 1K req/hari — cukup ~400 sampel/hari

In [1]:
# Install dependencies (jalankan sekali saja)
# !pip install groq rank-bm25 datasets

In [2]:
import os, sys, json, pickle, time, re, warnings
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict, Tuple
from pathlib import Path
from datetime import datetime

from groq import Groq
from rank_bm25 import BM25Okapi
from datasets import load_dataset

warnings.filterwarnings('ignore')
print('Semua library berhasil diimpor!')
print(f'Python: {sys.version.split()[0]} | NumPy: {np.__version__} | Pandas: {pd.__version__}')

C:\Users\Ricky Wijaya\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Semua library berhasil diimpor!
Python: 3.11.9 | NumPy: 2.3.5 | Pandas: 2.3.3


In [3]:
# ============================================================
# KONFIGURASI
# ============================================================

GROQ_API_KEY = os.environ.get('GROQ_API_KEY', 'YOUR_GROQ_KEY_HERE')

LLM_MODEL = 'meta-llama/llama-4-scout-17b-16e-instruct'  # Llama 4 Scout (17B MoE)

TOP_K_RETRIEVAL = 5

DATASET_NAME   = 'qiaojin/PubMedQA'
DATASET_SUBSET = 'pqa_labeled'
MAX_SAMPLES    = 500

TEMPERATURE = 0.0
SEED        = 42

NOTEBOOK_DIR    = Path('.')
BM25_INDEX_PATH = NOTEBOOK_DIR / 'pubmedqa_bm25.pkl'
RESULTS_DIR     = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)

CONFIG_NAME         = 'baseline_scout'
PHASE1_PATH         = RESULTS_DIR / f'{CONFIG_NAME}_phase1_answers.json'
PHASE2_CUSTOM_PATH  = RESULTS_DIR / f'{CONFIG_NAME}_phase2_custom.json'
FINAL_CSV_PATH      = RESULTS_DIR / f'{CONFIG_NAME}_results.csv'

print('Konfigurasi:')
print(f'  LLM        : {LLM_MODEL} (via Groq API)')
print(f'  Retriever  : BM25')
print(f'  Top-K      : {TOP_K_RETRIEVAL}')
print(f'  Max Sampel : {MAX_SAMPLES}')
print(f'  Config     : {CONFIG_NAME}')
print()
if 'YOUR_API_KEY' in GROQ_API_KEY:
    print('WARNING: GROQ_API_KEY belum diisi!')
else:
    print(f'GROQ_API_KEY: {GROQ_API_KEY[:8]}...{GROQ_API_KEY[-4:]}')

Konfigurasi:
  LLM        : meta-llama/llama-4-scout-17b-16e-instruct (via Groq API)
  Retriever  : BM25
  Top-K      : 5
  Max Sampel : 500
  Config     : baseline_scout

GROQ_API_KEY: gsk_kXgL...f7O0


In [4]:
@dataclass
class Document:
    text         : str
    pubid        : str
    question     : str
    section_label: str
    answer       : str
    decision     : str

@dataclass
class RetrievalResult:
    document: Document
    score   : float


def tokenize_bm25(text: str) -> List[str]:
    """Tokenizer untuk BM25: hapus tanda baca, lowercase, split spasi."""
    return re.sub(r'[^a-zA-Z0-9\s]', ' ', text.lower()).split()


sample_text = 'Does aspirin (75mg) reduce myocardial infarction risk?'
print(f'Tokenisasi BM25: {tokenize_bm25(sample_text)}')
print('Data classes dan tokenizer siap.')

Tokenisasi BM25: ['does', 'aspirin', '75mg', 'reduce', 'myocardial', 'infarction', 'risk']
Data classes dan tokenizer siap.


In [5]:
def load_pubmedqa(subset=DATASET_SUBSET, max_samples=MAX_SAMPLES):
    print(f'Memuat PubMedQA ({subset})...')
    dataset = load_dataset(DATASET_NAME, subset, trust_remote_code=True)
    data    = dataset['train']
    if max_samples and len(data) > max_samples:
        data = data.select(range(max_samples))
    print(f'Dimuat {len(data)} sampel')
    return data


def prepare_documents(data) -> List[Document]:
    docs = []
    for item in data:
        pubid = str(item['pubid'])
        for ctx, label in zip(item['context']['contexts'], item['context']['labels']):
            docs.append(Document(
                text=ctx.strip(), pubid=pubid,
                question=item['question'], section_label=label,
                answer=item['long_answer'], decision=item['final_decision']
            ))
    print(f'Total potongan dokumen: {len(docs)}')
    return docs


def load_or_build_bm25(data) -> Tuple[BM25Okapi, List[Document]]:
    """Muat BM25 index dari file jika ada, atau bangun dari scratch."""
    if BM25_INDEX_PATH.exists():
        print(f'Memuat BM25 index dari {BM25_INDEX_PATH}...')
        with open(BM25_INDEX_PATH, 'rb') as f:
            saved = pickle.load(f)
        print(f'Dimuat: {len(saved["documents"])} dokumen')
        return saved['bm25'], saved['documents']
    else:
        print('Membangun BM25 index baru...')
        documents = prepare_documents(data)
        tokenized = [tokenize_bm25(d.text) for d in documents]
        bm25      = BM25Okapi(tokenized)
        with open(BM25_INDEX_PATH, 'wb') as f:
            pickle.dump({'bm25': bm25, 'documents': documents}, f)
        print(f'Index disimpan ke {BM25_INDEX_PATH}')
        return bm25, documents


# Load dataset dengan MAX_SAMPLES=500 agar BM25 index kompatibel dengan notebook lain
# Evaluasi tetap hanya 50 sampel pertama
_full_data            = load_dataset(DATASET_NAME, DATASET_SUBSET, trust_remote_code=True)['train']
bm25_index, documents = load_or_build_bm25(_full_data.select(range(500)))
pubmedqa_data         = _full_data.select(range(MAX_SAMPLES))
print(f'\nEvaluasi akan menggunakan {len(pubmedqa_data)} sampel pertama.')

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Memuat BM25 index dari pubmedqa_bm25.pkl...
Dimuat: 1706 dokumen

Evaluasi akan menggunakan 500 sampel pertama.


In [6]:
# ============================================================
# Setup Groq Client + Utility
# ============================================================

groq_client = Groq(api_key=GROQ_API_KEY)

# Jeda antar request (30 RPM = 1 req per 2 detik)
GROQ_DELAY = 2.5  # detik antar request
_last_call_time = 0


def llm_generate(prompt: str, max_tokens: int = 300, temperature: float = TEMPERATURE) -> str:
    """
    Wrapper Groq API dengan:
    1. Jeda otomatis antar request (GROQ_DELAY) agar tidak kena rate limit
    2. Retry otomatis jika tetap kena 429
    """
    global _last_call_time

    # Jeda otomatis
    elapsed = time.time() - _last_call_time
    if elapsed < GROQ_DELAY:
        time.sleep(GROQ_DELAY - elapsed)

    for attempt in range(5):
        try:
            _last_call_time = time.time()
            response = groq_client.chat.completions.create(
                model=LLM_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=temperature,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate' in err.lower():
                wait = (attempt + 1) * 15
                print(f'  [Rate limit] Tunggu {wait}s... (attempt {attempt+1}/5)')
                time.sleep(wait)
            else:
                print(f'  [Groq Error] {type(e).__name__}: {err[:80]}')
                raise
    raise RuntimeError('Groq API gagal setelah 5 percobaan.')


# Smoke test
print('Testing Groq API (Llama 4 Scout)...')
_test = llm_generate('Reply with exactly: OK', max_tokens=5)
print(f'Response: {_test!r}')
print(f'Delay antar request: {GROQ_DELAY}s')
print('Groq client siap!')

Testing Groq API (Llama 4 Scout)...
Response: 'OK'
Delay antar request: 2.5s
Groq client siap!


In [7]:
def retrieve_baseline(query: str, k: int = TOP_K_RETRIEVAL) -> List[RetrievalResult]:
    """
    Retrieval baseline menggunakan BM25 (exact keyword matching).
    Identik dengan notebook 02 — tidak ada query rewriting, tidak ada reranking.
    """
    tokens = tokenize_bm25(query)
    scores = bm25_index.get_scores(tokens)
    top_k  = np.argsort(scores)[::-1][:k]
    return [RetrievalResult(document=documents[i], score=float(scores[i])) for i in top_k]


test_q = 'Does aspirin reduce the risk of myocardial infarction?'
test_r = retrieve_baseline(test_q)
print(f'Query: {test_q}')
print(f'Top-{TOP_K_RETRIEVAL} dokumen (BM25):')
for i, r in enumerate(test_r, 1):
    print(f'  [{i}] Score={r.score:.4f} | {r.document.section_label} | {r.document.text[:80]}...')

Query: Does aspirin reduce the risk of myocardial infarction?
Top-5 dokumen (BM25):
  [1] Score=23.4945 | DESIGN | Within a prospective, population-based cohort study individuals without history ...
  [2] Score=20.3167 | METHODS | By use of the Cooperative Cardiovascular Project database (a retrospective medic...
  [3] Score=19.3056 | METHODS | Of the 9681 women and 8888 men who attended risk assessment from 1967-1991, with...
  [4] Score=18.0483 | STUDY DESIGN | All patients between the ages of 80 to 89 years undergoing carotid endarterectom...
  [5] Score=17.2286 | OBJECTIVE | To examine the effect of a weekend hospitalization on the timing and incidence o...


In [8]:
# Prompt identik dengan notebook 02 — tidak ada perubahan
GENERATION_PROMPT = (
    'You are a medical research assistant. '
    'Answer a biomedical yes/no/maybe question based solely on the provided scientific abstracts.\n\n'
    'Context from medical literature:\n{context}\n\n'
    'Question: {question}\n\n'
    'Instructions:\n'
    '- Carefully read the context and assess whether it supports or refutes the question.\n'
    '- Provide a brief explanation (2-3 sentences) using ONLY the information above.\n'
    '- End your response with EXACTLY ONE of these words on its own line: yes, no, or maybe.\n'
    '  - yes   : the evidence supports the hypothesis, even if not perfectly conclusive\n'
    '  - no    : the evidence refutes or does not support the hypothesis\n'
    '  - maybe : ONLY if the evidence is directly contradictory (some findings say yes,\n'
    '            others say no), or if the context contains no relevant information at all\n'
    '- IMPORTANT: If the evidence leans in one direction, even partially, choose yes or no.\n'
    '  Do NOT use maybe simply because the evidence is limited or not 100%% certain.\n\n'
    'Answer:'
)


def generate_baseline_answer(query: str, retrieved: List[RetrievalResult]) -> str:
    """Generate jawaban via Groq. Baseline: tanpa QR, tanpa reranking."""
    context = '\n\n'.join(
        f'[{i}] ({r.document.section_label}): {r.document.text}'
        for i, r in enumerate(retrieved, 1)
    )
    return llm_generate(
        GENERATION_PROMPT.format(context=context, question=query),
        max_tokens=300,
        temperature=TEMPERATURE
    )


print('Testing generation...')
test_ans = generate_baseline_answer(test_q, test_r)
print('-' * 60)
print(test_ans)
print('-' * 60)

Testing generation...
------------------------------------------------------------
The provided abstracts do not directly mention the effect of aspirin on the risk of myocardial infarction. However, several studies mentioned assess risk factors and outcomes related to myocardial infarction, but none explicitly discuss aspirin as a preventive measure. Given that the context does not provide direct information about aspirin's effect on myocardial infarction risk, we cannot conclusively determine its impact.

maybe
------------------------------------------------------------


In [9]:
def extract_label(answer: str) -> str:
    """
    Ekstrak prediksi yes/no/maybe dari teks jawaban.
    Strategi:
      1. Kata standalone di 3 baris terakhir
      2. Kata standalone di seluruh teks
      3. Default ke 'maybe'
    """
    lines = [l.strip().lower() for l in answer.split('\n') if l.strip()]
    for line in reversed(lines[-3:]):
        word = re.sub(r'[^a-z]', '', line)
        if word in ('yes', 'no', 'maybe'):
            return word
    for label in ('yes', 'no', 'maybe'):
        if re.search(r'\b' + label + r'\b', answer.lower()):
            return label
    return 'maybe'


# Unit test
cases = [
    ('Strong evidence.\nyes', 'yes'),
    ('No effect found.\nno',  'no'),
    ('Mixed results.\nmaybe', 'maybe'),
    ('Verdict: yes.',          'yes'),
    ('Totally unclear.',       'maybe'),
]
all_ok = all(extract_label(txt) == exp for txt, exp in cases)
print(f'Unit test extract_label: {"PASS" if all_ok else "FAIL"}')
print(f'Label dari test answer: {extract_label(test_ans)!r}')

Unit test extract_label: PASS
Label dari test answer: 'maybe'


In [10]:
# ============================================================
# Custom Zero-NaN Evaluator (pakai Groq, bukan Ollama)
# Logika identik dengan notebook 02 — hanya _llm_yes_no yang diganti
# ============================================================

def _split_sentences(text: str) -> List[str]:
    """Pecah teks menjadi kalimat. Filter kalimat terlalu pendek (<15 char)."""
    parts = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in parts if len(s.strip()) >= 15]


def _llm_yes_no(prompt: str) -> bool:
    """Tanya Groq LLM ya/tidak. Return True=yes, False=no. Fallback False jika error."""
    try:
        resp = llm_generate(prompt, max_tokens=10, temperature=0.0)
        return 'yes' in resp.lower()[:15]
    except Exception:
        return False


def compute_faithfulness(answer: str, contexts: List[str]) -> float:
    """
    Faithfulness: fraksi kalimat jawaban yang didukung konteks.
    Selalu return 0.0–1.0, tidak pernah NaN.
    """
    sentences = _split_sentences(answer)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement directly supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    supported = sum(
        1 for s in sentences
        if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s))
    )
    return supported / len(sentences)


def compute_context_recall(reference: str, contexts: List[str]) -> float:
    """
    Context Recall: fraksi fakta di reference yang tercakup konteks.
    Selalu return 0.0–1.0, tidak pernah NaN.
    """
    sentences = _split_sentences(reference)
    if not sentences:
        return 0.0
    ctx_text = '\n'.join(f'[{i+1}] {c[:400]}' for i, c in enumerate(contexts))
    prompt_tmpl = (
        'Context:\n{ctx}\n\n'
        'Statement: {sent}\n\n'
        'Is this statement supported by the context above? '
        'Answer with only "yes" or "no".'
    )
    covered = sum(
        1 for s in sentences
        if _llm_yes_no(prompt_tmpl.format(ctx=ctx_text, sent=s))
    )
    return covered / len(sentences)


def evaluate_custom(question: str, answer: str,
                    contexts: List[str], reference: str) -> Dict:
    """Wrapper evaluasi 1 sampel. Selalu return dict tanpa NaN."""
    return {
        'faithfulness'  : compute_faithfulness(answer, contexts),
        'context_recall': compute_context_recall(reference, contexts),
    }


# Smoke test
_ctx = ['Aspirin reduces blood clotting and is used for heart attack prevention.']
_ans = 'Aspirin helps prevent heart attacks. It works by reducing clotting.'
_ref = 'Aspirin is used for heart attack prevention by reducing blood clotting.'
_r   = evaluate_custom('Does aspirin prevent heart attacks?', _ans, _ctx, _ref)
print(f'Smoke test evaluator:')
print(f'  faithfulness   = {_r["faithfulness"]:.3f}')
print(f'  context_recall = {_r["context_recall"]:.3f}')
print('Zero-NaN evaluator siap (via Groq).')

Smoke test evaluator:
  faithfulness   = 1.000
  context_recall = 1.000
Zero-NaN evaluator siap (via Groq).


## Demo — 5 Sampel Pertama

In [ ]:
DEMO_SIZE    = 5
demo_results = []

print(f'DEMO: {DEMO_SIZE} sampel pertama (Groq: {LLM_MODEL})')
print('=' * 65)

for i in range(DEMO_SIZE):
    s         = pubmedqa_data[i]
    q         = s['question']
    gt        = s['final_decision']

    retrieved = retrieve_baseline(q)
    answer    = generate_baseline_answer(q, retrieved)
    predicted = extract_label(answer)
    correct   = predicted == gt

    demo_results.append({
        'idx': i, 'question': q,
        'ground_truth': gt, 'predicted_label': predicted,
        'is_correct': correct, 'answer': answer,
    })

    verdict = '✅ BENAR' if correct else '❌ SALAH'
    print(f'\n[{i+1}/{DEMO_SIZE}] {q[:75]}...')
    print(f'  GT={gt} | Pred={predicted} | {verdict}')
    print(f'  Jawaban: {answer[:120]}...')

n_ok = sum(r['is_correct'] for r in demo_results)
print(f'\n{"="*65}')
print(f'Demo Accuracy: {n_ok}/{DEMO_SIZE} = {n_ok/DEMO_SIZE:.1%}')

## Phase 1 — Generate Jawaban (50 Sampel)

Estimasi waktu dengan Groq free tier: **~2–3 menit** untuk 50 sampel.
Resume otomatis jika interrupted.

In [11]:
if PHASE1_PATH.exists():
    with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
        phase1_results = json.load(f)['results']
    start_from = len(phase1_results)
    print(f'Resume Phase 1: {start_from}/{MAX_SAMPLES} sudah selesai.')
else:
    phase1_results, start_from = [], 0
    print(f'Mulai Phase 1: {MAX_SAMPLES} sampel.')

if start_from < MAX_SAMPLES:
    print(f'Memproses {MAX_SAMPLES - start_from} sampel tersisa...\n')
    t_start = time.time()

    for i in range(start_from, MAX_SAMPLES):
        s          = pubmedqa_data[i]
        q, gt, ref = s['question'], s['final_decision'], s['long_answer']

        retrieved  = retrieve_baseline(q)
        answer     = generate_baseline_answer(q, retrieved)
        predicted  = extract_label(answer)

        phase1_results.append({
            'idx'            : i,
            'pubid'          : str(s['pubid']),
            'question'       : q,
            'ground_truth'   : gt,
            'predicted_label': predicted,
            'is_correct'     : predicted == gt,
            'answer'         : answer,
            'contexts'       : [r.document.text for r in retrieved],
            'reference'      : ref,
            'retrieval_scores': [r.score for r in retrieved],
        })

        # Simpan setiap 10 sampel (resume-able)
        if (i + 1) % 10 == 0 or i == MAX_SAMPLES - 1:
            with open(PHASE1_PATH, 'w', encoding='utf-8') as f:
                json.dump({
                    'config'    : CONFIG_NAME,
                    'llm_model' : LLM_MODEL,
                    'timestamp' : datetime.now().isoformat(),
                    'max_samples': MAX_SAMPLES,
                    'completed' : i + 1,
                    'results'   : phase1_results
                }, f, indent=2, ensure_ascii=False)
            done = i + 1
            acc  = sum(r['is_correct'] for r in phase1_results) / done
            eta  = (time.time()-t_start) / done * (MAX_SAMPLES-done) / 60
            print(f'  [{done:3d}/{MAX_SAMPLES}] Accuracy: {acc:.1%} | pred={predicted}, gt={gt} | ETA {eta:.1f} mnt')

    print(f'\nPhase 1 selesai! -> {PHASE1_PATH}')
else:
    print(f'Phase 1 sudah selesai ({MAX_SAMPLES} sampel).')

Mulai Phase 1: 500 sampel.
Memproses 500 sampel tersisa...

  [ 10/500] Accuracy: 50.0% | pred=yes, gt=yes | ETA 19.1 mnt
  [ 20/500] Accuracy: 70.0% | pred=yes, gt=yes | ETA 19.3 mnt
  [ 30/500] Accuracy: 73.3% | pred=yes, gt=yes | ETA 19.2 mnt
  [ 40/500] Accuracy: 65.0% | pred=yes, gt=no | ETA 19.0 mnt
  [ 50/500] Accuracy: 64.0% | pred=maybe, gt=no | ETA 18.7 mnt
  [ 60/500] Accuracy: 60.0% | pred=yes, gt=yes | ETA 18.3 mnt
  [ 70/500] Accuracy: 61.4% | pred=yes, gt=yes | ETA 17.9 mnt
  [ 80/500] Accuracy: 62.5% | pred=yes, gt=yes | ETA 17.5 mnt
  [ 90/500] Accuracy: 63.3% | pred=no, gt=maybe | ETA 17.1 mnt
  [100/500] Accuracy: 66.0% | pred=yes, gt=yes | ETA 16.7 mnt
  [110/500] Accuracy: 67.3% | pred=yes, gt=yes | ETA 16.2 mnt
  [120/500] Accuracy: 65.8% | pred=yes, gt=yes | ETA 15.8 mnt
  [130/500] Accuracy: 63.1% | pred=yes, gt=maybe | ETA 15.4 mnt
  [140/500] Accuracy: 62.9% | pred=yes, gt=yes | ETA 15.0 mnt
  [150/500] Accuracy: 62.7% | pred=yes, gt=no | ETA 14.6 mnt
  [160/5

## Analisis Phase 1

In [11]:
with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    results_p1 = json.load(f)['results']

n         = len(results_p1)
n_correct = sum(r['is_correct'] for r in results_p1)
gts       = [r['ground_truth']    for r in results_p1]
preds     = [r['predicted_label'] for r in results_p1]

print(f'ANALISIS PHASE 1 — {n} sampel ({CONFIG_NAME})')
print('=' * 55)
print(f'Label Accuracy    : {n_correct}/{n} = {n_correct/n:.1%}')
print(f'Hallucination Rate: {(n-n_correct)/n:.1%}\n')

print(f'  {"Label":<8} | {"Ground Truth":>12} | {"Prediksi":>10}')
print(f'  {"-"*40}')
for lbl in ['yes','no','maybe']:
    g, p = gts.count(lbl), preds.count(lbl)
    print(f'  {lbl:<8} | {g:>6} ({g/n:.0%})    | {p:>6} ({p/n:.0%})')

print('\nConfusion Matrix (baris=GT, kolom=Pred):')
lbls = ['yes','no','maybe']
print('  ' + f'{"GT/Pred":>8}' + ''.join(f'{l:>8}' for l in lbls))
for gt_l in lbls:
    row = f'  {gt_l:>8}'
    for pr_l in lbls:
        cnt = sum(1 for r in results_p1 if r['ground_truth']==gt_l and r['predicted_label']==pr_l)
        row += f'{cnt:>8}'
    print(row)

# Bandingkan dengan baseline Ollama (notebook 02) jika ada
baseline_p1 = Path('../results/baseline_phase1_answers.json')
if baseline_p1.exists():
    with open(baseline_p1) as f:
        bl = json.load(f)['results'][:n]
    bl_acc = sum(r['is_correct'] for r in bl) / len(bl)
    print(f'\n--- Perbandingan dengan Baseline Ollama (llama3.2) ---')
    print(f'  Baseline Ollama (llama3.2)     : {bl_acc:.1%}')
    print(f'  Baseline Groq   ({LLM_MODEL[:20]}): {n_correct/n:.1%}')
    print(f'  Delta                          : {(n_correct/n - bl_acc):+.1%}')

ANALISIS PHASE 1 — 500 sampel (baseline_scout)
Label Accuracy    : 333/500 = 66.6%
Hallucination Rate: 33.4%

  Label    | Ground Truth |   Prediksi
  ----------------------------------------
  yes      |    275 (55%)    |    339 (68%)
  no       |    159 (32%)    |    123 (25%)
  maybe    |     66 (13%)    |     38 (8%)

Confusion Matrix (baris=GT, kolom=Pred):
   GT/Pred     yes      no   maybe
       yes     240      22      13
        no      55      86      18
     maybe      44      15       7

--- Perbandingan dengan Baseline Ollama (llama3.2) ---
  Baseline Ollama (llama3.2)     : 55.6%
  Baseline Groq   (meta-llama/llama-4-s): 66.6%
  Delta                          : +11.0%


## Phase 2 — Custom Evaluator (50 Sampel)

Estimasi waktu: **~10–15 menit** untuk 50 sampel (banyak LLM call per sampel).
Resume otomatis jika interrupted.

In [ ]:
MAX_CUSTOM_SAMPLES = MAX_SAMPLES  # 50

with open(PHASE1_PATH, 'r', encoding='utf-8') as f:
    p1_custom = json.load(f)['results'][:MAX_CUSTOM_SAMPLES]

if PHASE2_CUSTOM_PATH.exists():
    with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
        p2_custom = json.load(f)['results']
    done_custom = {r['idx'] for r in p2_custom}
    print(f'Resume Phase 2: {len(done_custom)}/{MAX_CUSTOM_SAMPLES} selesai.')
else:
    p2_custom, done_custom = [], set()
    print(f'Mulai Phase 2: {MAX_CUSTOM_SAMPLES} sampel (custom zero-NaN).')

remaining = [r for r in p1_custom if r['idx'] not in done_custom]
print(f'Sisa: {len(remaining)} sampel\n')

t0 = time.time()
for i, r in enumerate(remaining):
    scores = evaluate_custom(r['question'], r['answer'], r['contexts'], r['reference'])
    p2_custom.append({
        'idx'            : r['idx'],
        'ground_truth'   : r['ground_truth'],
        'predicted_label': r['predicted_label'],
        'is_correct'     : r['is_correct'],
        **scores
    })

    # Simpan setiap 5 sampel
    if (i + 1) % 5 == 0 or i == len(remaining) - 1:
        with open(PHASE2_CUSTOM_PATH, 'w', encoding='utf-8') as f:
            json.dump({
                'config'    : CONFIG_NAME,
                'llm_model' : LLM_MODEL,
                'timestamp' : datetime.now().isoformat(),
                'max_samples': MAX_CUSTOM_SAMPLES,
                'metrics'   : ['faithfulness', 'context_recall'],
                'evaluator' : 'custom_zero_nan',
                'results'   : p2_custom
            }, f, indent=2, ensure_ascii=False)
        done  = i + 1
        total = len(remaining)
        eta   = (time.time()-t0)/done*(total-done)/60 if done < total else 0
        avg_f = sum(x['faithfulness']   for x in p2_custom) / len(p2_custom)
        avg_r = sum(x['context_recall'] for x in p2_custom) / len(p2_custom)
        print(f'  [{done:3d}/{total}] idx={r["idx"]} | '
              f'faith={scores["faithfulness"]:.3f} | cr={scores["context_recall"]:.3f} | '
              f'avg_f={avg_f:.3f} | avg_cr={avg_r:.3f} | ETA {eta:.1f} mnt')

print(f'\nPhase 2 selesai! -> {PHASE2_CUSTOM_PATH}')

Mulai Phase 2: 500 sampel (custom zero-NaN).
Sisa: 500 sampel

  [  5/500] idx=4 | faith=1.000 | cr=0.833 | avg_f=0.600 | avg_cr=0.600 | ETA 124.3 mnt
  [ 10/500] idx=9 | faith=0.000 | cr=0.000 | avg_f=0.667 | avg_cr=0.567 | ETA 112.7 mnt
  [ 15/500] idx=14 | faith=1.000 | cr=1.000 | avg_f=0.711 | avg_cr=0.622 | ETA 109.4 mnt
  [ 20/500] idx=19 | faith=0.667 | cr=1.000 | avg_f=0.667 | avg_cr=0.700 | ETA 103.2 mnt
  [ 25/500] idx=24 | faith=1.000 | cr=1.000 | avg_f=0.720 | avg_cr=0.713 | ETA 103.5 mnt
  [ 30/500] idx=29 | faith=0.667 | cr=1.000 | avg_f=0.700 | avg_cr=0.694 | ETA 99.5 mnt
  [ 35/500] idx=34 | faith=1.000 | cr=1.000 | avg_f=0.733 | avg_cr=0.710 | ETA 99.1 mnt
  [ 40/500] idx=39 | faith=0.667 | cr=1.000 | avg_f=0.750 | avg_cr=0.708 | ETA 97.0 mnt
  [ 45/500] idx=44 | faith=1.000 | cr=1.000 | avg_f=0.733 | avg_cr=0.685 | ETA 95.4 mnt
  [ 50/500] idx=49 | faith=0.667 | cr=0.000 | avg_f=0.743 | avg_cr=0.657 | ETA 94.0 mnt
  [ 55/500] idx=54 | faith=1.000 | cr=0.000 | avg_f=0.

## Summary — Hasil Akhir

In [ ]:
with open(PHASE2_CUSTOM_PATH, 'r', encoding='utf-8') as f:
    p2 = json.load(f)['results']

n      = len(p2)
acc    = sum(r['is_correct']     for r in p2) / n
avg_f  = sum(r['faithfulness']   for r in p2) / n
avg_cr = sum(r['context_recall'] for r in p2) / n

print('=' * 60)
print(f'  {CONFIG_NAME.upper()} — {n} sampel')
print(f'  LLM: {LLM_MODEL}')
print('=' * 60)
print(f'  Label Accuracy    : {acc:.1%}')
print(f'  Hallucination Rate: {1-acc:.1%}')
print(f'  Faithfulness      : {avg_f:.4f}')
print(f'  Context Recall    : {avg_cr:.4f}')
print(f'  NaN count         : 0 (dijamin custom evaluator)')
print('=' * 60)

# Per-label
print('\nPer-label accuracy:')
for lbl in ['yes','no','maybe']:
    sub = [r for r in p2 if r['ground_truth'] == lbl]
    if sub:
        lbl_acc  = sum(r['is_correct']   for r in sub) / len(sub)
        lbl_f    = sum(r['faithfulness'] for r in sub) / len(sub)
        lbl_cr   = sum(r['context_recall'] for r in sub) / len(sub)
        print(f'  {lbl:>5}: acc={lbl_acc:.1%} (n={len(sub)}) | faith={lbl_f:.3f} | ctx_recall={lbl_cr:.3f}')

# Bandingkan dengan baseline Ollama
bl_custom = Path('../results/baseline_phase2_custom.json')
if bl_custom.exists():
    with open(bl_custom) as f:
        bl_p2 = json.load(f)['results'][:n]
    bl_acc = sum(r['is_correct']     for r in bl_p2) / len(bl_p2)
    bl_f   = sum(r['faithfulness']   for r in bl_p2) / len(bl_p2)
    bl_cr  = sum(r['context_recall'] for r in bl_p2) / len(bl_p2)
    print(f'\n--- Perbandingan dengan Baseline Ollama (llama3.2) ---')
    print(f'  {"Metrik":<22} | {"Ollama llama3.2":>16} | {"Groq llama3.3-70b":>17} | {"Delta":>8}')
    print(f'  {"-"*70}')
    for metrik, bv, gv in [
        ('Label Accuracy', bl_acc, acc),
        ('Faithfulness',   bl_f,   avg_f),
        ('Context Recall', bl_cr,  avg_cr),
    ]:
        delta = gv - bv
        tanda = '↑' if delta > 0.001 else ('↓' if delta < -0.001 else '=')
        print(f'  {metrik:<22} | {bv:>16.4f} | {gv:>17.4f} | {delta:>+7.4f} {tanda}')

print(f'\nBaris tabel skripsi (Groq):')
print(f'  | Baseline Groq (llama-3.3-70b) | {acc:.3f} | {1-acc:.3f} | {avg_f:.3f} | {avg_cr:.3f} |')